## Crossover sensitivity

**Calculate percentage of patients that are high risk at different crossover thresholds: 14d, 30d, and 60dd**

In [1]:
import numpy as np
import pandas as pd

## Import data

In [2]:
treatment_df = pd.read_csv('../outputs/ioio_tki_index.csv')

In [3]:
treatment_df.sample(3)

,PatientID,LineName,StartDate
1688,F018BC705E466,ioio,2020-08-17
1011,F0B5918C28C33,ioio,2018-10-08
590,F875ABB8AFCDB,ioio,2021-09-02


In [4]:
treatment_df.shape

(2771, 3)

In [5]:
treatment_df['treatment'] = (treatment_df['LineName'] == 'ioio').astype(int)

In [6]:
dtype_map = pd.read_csv('../outputs/ioio_tki_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/ioio_tki_features_df.csv', dtype = dtype_map)

In [7]:
features_df.head(3)

,PatientID,CalcResectInitialDx,TStage_mod,NStage_mod,MStage_mod,GroupStage_mod,ResidualDiseaseInitialDx_mod,ResidualDiseaseLocalRecur_mod,CalcResectLocalRecur_mod,days_diagnosis_to_adv,...,primary_site_procedure,lymph_node_procedure,other_viscera_met,thoracic_met,other_met,lymph_met,skin_met,bone_met,brain_met,liver_met
0,F744F618949B5,NaN,NaN,NaN,NaN,4.0,NaN,unknown,Unknown,0.0,...,0,0,1,1,1,0,1,0,0,0
1,F4AAE7EB8AE49,NaN,NaN,NaN,NaN,4.0,NaN,unknown,Unknown,0.0,...,0,0,0,0,0,0,0,1,0,1
2,F702FE1F825B7,Resectable,T3,N1,M0,3.0,R0,R0,Resectable,0.0,...,1,1,0,0,0,0,0,0,0,0


In [8]:
features_df.shape

(1339, 174)

In [9]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [10]:
df.shape

(1339, 177)

In [11]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [12]:
surv_pred_df.shape

(1069, 2)

In [13]:
df = pd.merge(df, surv_pred_df, on = 'PatientID', how = 'left')

In [14]:
df.shape

(1339, 178)

In [15]:
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [16]:
df['treatment_year'] = df['StartDate'].dt.year

In [17]:
df = df.query('treatment_year <= 2021')

In [18]:
df.shape

(1069, 179)

In [19]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate_30 = float(f.read())

with open('../outputs/crossover_survival_estimate_14.txt', 'r') as f:
    crossover_survival_estimate_14 = float(f.read())

with open('../outputs/crossover_survival_estimate_60.txt', 'r') as f:
    crossover_survival_estimate_60 = float(f.read())

In [20]:
print(f'r* for 14d crossover: {crossover_survival_estimate_14}')
print(f'r* for 30d crossover: {crossover_survival_estimate_30}')
print(f'r* for 60d crossover: {crossover_survival_estimate_60}')

r* for 14d crossover: -3.0364566162750193
r* for 30d crossover: -1.734162883030624
r* for 60d crossover: 0.7076378668026176


In [21]:
print(f'percent high risk at 14d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_14').shape[0]/df.shape[0]}')
print(f'percent high risk at 30d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_30').shape[0]/df.shape[0]}')
print(f'percent high risk at 60d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_60').shape[0]/df.shape[0]}')

percent high risk at 14d crossover: 0.0
percent high risk at 30d crossover: 0.0
percent high risk at 60d crossover: 0.33489242282507015
